In [ ]:
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np
from multiprocessing import Pool, cpu_count
import os
from scipy.optimize import curve_fit
from scipy.special import gamma

# Study murmurations of families of elliptic curves with non-trivial MW torsion, modulo quadratic twists.

# =============================================================================
# CONFIGURATION
# =============================================================================
NMIN = -300
NMAX = 300

# Load curve database once
try:
    curve_db
    print(f"curve_db loaded with {len(curve_db)} isogeny classes")
except NameError:
    print("Loading curve database...")
    curve_db = load('Curve Database (Conductor < 100 000)/curve_database_c1_to_100000.sobj')
    print(f"Loaded curve_db with {len(curve_db)} isogeny classes")

# Elliptic curve parameterizations
def elliptic_curve_from_phi_Z2(phi): 
    # take phi=a4/a2^2
    a2 = 1
    
    a = a2**2*(phi-1/3)
    b = 1/27*a2**3*(2-9*phi)
    return a, b

def elliptic_curve_from_phi_Z3(phi):
    #phi = a3 / a1**3, a1=1
    # Notice that since we get factors of a1^4 and a1^6 in a and b respectively, we can drop them
    # by the usual rescaling (x,y)->(l^2x,l^3y), effectively giving a one-parameter family of curves.
    # Notice that this cannot be done for torsion Z2 and Z2+

    a1 = 1
    a = a1**4 * (phi/2 - 1/48)
    b = a1**6 * (phi**2/4 - phi/24 + 1/864)
    return a, b

def elliptic_curve_from_phi_Z4(phi):
    # phi = a2 / a1**2

    a1 = 1
    a = a1**4 * (-phi**2/3 + phi/3 - 1/48)
    b = a1**6 * (2*phi**3/27 + 5*phi**2/36 - phi/36 + 1/864)
    return a, b

def elliptic_curve_from_phi_Z5(phi):
    # phi = b1 / a1
    a1 = 1

    a = a1**4 * (-phi**4/3 + phi**3/6 + phi**2/3 - phi/6 - 1/48)
    b = a1**6/864 * (1 - 2*phi + 2*phi**2) * (1 + 14*phi + 26*phi**2 - 116*phi**3 + 76*phi**4)
    return a, b

def elliptic_curve_from_phi_Z6(phi):
    # phi = b1 / a1
    a1 = 1

    a = (a1**4 / 192) * phi * (3 - 3*phi - 3*phi**2 - phi**3)
    b = (a1**6 / 110592) * (3 - 6*phi - phi**2) * (9 - 6*phi**2 - 24*phi**3 - 11*phi**4)
    return a, b

def elliptic_curve_from_phi_Z2Z2(phi):
    b2 = 1
    a = (b2**2 / 3) * (phi - 1 - phi**2)
    b = -(b2**3 / 27) * (1 + phi) * (1 - 2*phi) * (2 - phi)
    return a, b

def elliptic_curve_from_phi_Z4Z2(phi):
    a1 = 1
    a = a1**4 * (-1/768 - (7/24)*phi**2 - (1/3)*phi**4)
    b = (a1**6 / 55296) * (1 + 16*phi**2) * (1 - 24*phi + 16*phi**2) * (1 + 24*phi + 16*phi**2)
    return a, b

def power_law(x, A, alpha):
    return A / (x**alpha)

def student_t_norm(x, x0, sigma, nu):
    norm = gamma((nu + 1) / 2) / (np.sqrt(nu * np.pi) * sigma * gamma(nu / 2))
    return norm * (1 + ((x - x0)**2) / (nu * sigma**2)) ** (-(nu + 1) / 2)

def get_isogeny_class(label):
    i = len(label) - 1
    while i >= 0 and label[i].isdigit():
        i -= 1
    return label[:i+1]

def generate_phis_for_m(args):
    m, nmin, nmax = args
    return [QQ(m/n) for n in range(nmin, nmax) if n != 0]

# Global variable for multiprocessing
_elliptic_curve_from_phi = None

def process_phi(phi):
    try:
        a, b = _elliptic_curve_from_phi(phi)
        E = EllipticCurve([0, 0, 0, a, b])
        label = E.label()
        iso = get_isogeny_class(label)
        return (phi, curve_db[iso]['conductor'], iso)
    except:
        return None

def run_torsion_scan(torsion_type, nmin=NMIN, nmax=NMAX):
    global _elliptic_curve_from_phi
    
    torsion_map = {
        'Z2': (elliptic_curve_from_phi_Z2, 'MW_torsion_curves/Z_2'),
        'Z3': (elliptic_curve_from_phi_Z3, 'MW_torsion_curves/Z_3'),
        'Z4': (elliptic_curve_from_phi_Z4, 'MW_torsion_curves/Z_4'),
        'Z5': (elliptic_curve_from_phi_Z5, 'MW_torsion_curves/Z_5'),
        'Z6': (elliptic_curve_from_phi_Z6, 'MW_torsion_curves/Z_6'),
        'Z2+Z2': (elliptic_curve_from_phi_Z2Z2, 'MW_torsion_curves/Z_2+Z_2'),
        'Z4+Z2': (elliptic_curve_from_phi_Z4Z2, 'MW_torsion_curves/Z_4+Z_2'),
    }
    
    if torsion_type not in torsion_map:
        raise ValueError(f"Unknown torsion type: {torsion_type}")
    
    _elliptic_curve_from_phi, output_dir = torsion_map[torsion_type]
    
    os.makedirs(output_dir, exist_ok=True)
    print(f"\n{'='*60}")
    print(f"SCANNING {torsion_type} TORSION FAMILY")
    print(f"{'='*60}")
    
    # Generate phis
    print("Generating phis...")
    m_values = [(m, nmin, nmax) for m in range(nmin, nmax)]
    with Pool(processes=cpu_count()) as pool:
        phi_lists = list(tqdm(pool.imap(generate_phis_for_m, m_values), total=len(m_values)))
    phis = list(set([phi for sublist in phi_lists for phi in sublist]))
    print(f"Generated {len(phis)} unique phis")
    
    # Process phis
    print("Processing phis...")
    with Pool(processes=cpu_count()) as pool:
        results = list(tqdm(pool.imap(process_phi, phis), total=len(phis)))
    
    # Track multiplicities
    print("Analyzing multiplicities...")
    phi_to_iso, iso_to_count, iso_to_phis = {}, {}, {}
    for phi, result in zip(phis, results):
        if result is not None:
            _, _, iso = result
            phi_to_iso[phi] = iso
            iso_to_count[iso] = iso_to_count.get(iso, 0) + 1
            iso_to_phis.setdefault(iso, []).append(phi)
        else:
            phi_to_iso[phi] = None
    
    multiplicities = [0 if phi_to_iso.get(phi) is None else iso_to_count[phi_to_iso[phi]] for phi in phis]
    
    # Filter to unique isogeny classes
    cs, seen_iso_classes, phi_reps = [], [], []
    for result in results:
        if result is not None:
            phi, c, iso = result
            if iso not in seen_iso_classes:
                cs.append(c)
                seen_iso_classes.append(iso)
                phi_reps.append(phi)
    
    print(f"Found {len(phi_reps)} distinct isogeny classes with c < 100,000")
    
    # Save results
    print("Saving data...")
    scan_data = {
        'phi_reps': phi_reps, 'conductors': cs, 'isogeny_classes': seen_iso_classes,
        'all_phis': phis, 'multiplicities': multiplicities, 'phi_to_iso': phi_to_iso,
        'iso_to_count': iso_to_count, 'iso_to_phis': iso_to_phis,
        'nmin': nmin, 'nmax': nmax
    }
    save(scan_data, f'{output_dir}/({nmin},{nmax})_phi_scan_data.sobj')
    
    # Plotting
    print("Creating plots...")
    phis_float = [float(phi) for phi in phis]
    phi_reps_float = [float(phi) for phi in phi_reps]
    
    # Plot 1: Multiplicity vs phi
    plt.figure(figsize=(14, 7))
    scatter = plt.scatter(phis_float, multiplicities, c=multiplicities, cmap='plasma', alpha=0.6, s=20, edgecolors='black', linewidth=0.3)
    plt.colorbar(scatter, label='Multiplicity')
    plt.xlabel(r'$\phi$', fontsize=14); plt.ylabel('Multiplicity', fontsize=14)
    plt.title(fr'{torsion_type}: Isogeny Class Multiplicity vs $\phi$', fontsize=16)
    plt.grid(True, alpha=0.3); plt.tight_layout()
    plt.savefig(f'{output_dir}/({nmin},{nmax})_multiplicity_vs_phi.png', dpi=150); plt.show(); plt.close()
    
    # Plot 2: Phi distribution with Student's t fit
    plt.figure(figsize=(12, 6))
    counts_phi, bins_phi, _ = plt.hist(phi_reps_float, bins=100, density=True, edgecolor='black', alpha=0.7, color='steelblue', label='Data')
    bin_centers_phi = 0.5 * (bins_phi[:-1] + bins_phi[1:])
    try:
        popt, _ = curve_fit(student_t_norm, bin_centers_phi[counts_phi > 0], counts_phi[counts_phi > 0],
                            p0=[np.mean(phi_reps_float), np.std(phi_reps_float), 2.0], bounds=([-np.inf, 1e-6, 0.1], [np.inf, np.inf, 100]), maxfev=20000)
        x_fit = np.linspace(min(phi_reps_float), max(phi_reps_float), 1000)
        plt.plot(x_fit, student_t_norm(x_fit, *popt), 'r-', lw=2, label=fr"Student-$t$: $x_0$={popt[0]:.3f}, $\sigma$={popt[1]:.3f}, $\nu$={popt[2]:.2f}")
    except: pass
    plt.xlabel(r'$\phi$', fontsize=13); plt.ylabel('Density', fontsize=13)
    plt.title(fr'{torsion_type}: Distribution of $\phi$ ({len(phi_reps)} unique)', fontsize=14)
    plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
    plt.savefig(f'{output_dir}/({nmin},{nmax})_phi_dist_tfit.png', dpi=150); plt.show(); plt.close()
    
    # Plot 3: Conductor vs phi
    plt.figure(figsize=(12, 7))
    scatter = plt.scatter(phi_reps, cs, c=cs, cmap='viridis', alpha=0.6, s=25, edgecolors='black', linewidth=0.3)
    plt.colorbar(scatter, label='Conductor')
    plt.xlabel(r'$\phi$', fontsize=14); plt.ylabel('Conductor', fontsize=14)
    plt.title(fr'{torsion_type}: Conductor vs $\phi$', fontsize=16)
    plt.grid(True, alpha=0.3); plt.tight_layout()
    plt.savefig(f'{output_dir}/({nmin},{nmax})_conductor_vs_phi.png', dpi=150); plt.show(); plt.close()
    
    # Plot 4: Conductor distribution with power law fit
    plt.figure(figsize=(14, 6))
    counts_c, bins_c, _ = plt.hist(cs, bins=100, density=True, alpha=0.6, label='Data')
    bin_centers_c = (bins_c[:-1] + bins_c[1:]) / 2
    try:
        popt_c, _ = curve_fit(power_law, bin_centers_c[counts_c > 0], counts_c[counts_c > 0], p0=[1e6, 1.5], maxfev=10000)
        x_fit_c = np.linspace(min(cs), max(cs), 1000)
        plt.plot(x_fit_c, power_law(x_fit_c, *popt_c), 'r-', lw=2, label=f'Power law: α={popt_c[1]:.3f}')
    except: pass
    plt.ylim(0, np.max(counts_c) * 1.1)
    plt.xlabel("Conductor", fontsize=12); plt.ylabel("Density", fontsize=12)
    plt.title(fr'{torsion_type}: Conductor distribution', fontsize=14)
    plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
    plt.savefig(f'{output_dir}/({nmin},{nmax})_conductor_distribution_fit.png', dpi=150); plt.show(); plt.close()
    
    # Plot 5: Log-log conductor distribution
    plt.figure(figsize=(14, 6))
    plt.loglog(bin_centers_c[counts_c > 0], counts_c[counts_c > 0], 'bo', alpha=0.6, label='Data', markersize=4)
    try:
        plt.loglog(x_fit_c[x_fit_c > 0], power_law(x_fit_c[x_fit_c > 0], *popt_c), 'r-', lw=2, label=f'Power law: α={popt_c[1]:.3f}')
    except: pass
    plt.xlabel("Conductor (log)", fontsize=12); plt.ylabel("Density (log)", fontsize=12)
    plt.title(fr'{torsion_type}: Log-log conductor distribution', fontsize=14)
    plt.legend(); plt.grid(True, alpha=0.3, which='both'); plt.tight_layout()
    plt.savefig(f'{output_dir}/({nmin},{nmax})_conductor_distribution_loglog.png', dpi=150); plt.show(); plt.close()
    
    # Plot 6: Multiplicity histogram
    mult_counts = {}
    for m in multiplicities:
        mult_counts[m] = mult_counts.get(m, 0) + 1
    mult_values = sorted([m for m in mult_counts.keys() if m > 0])
    plt.figure(figsize=(12, 6))
    plt.bar(mult_values, [mult_counts[m] for m in mult_values], color='teal', edgecolor='black', alpha=0.7)
    plt.xlabel('Multiplicity', fontsize=14); plt.ylabel('Count', fontsize=14)
    plt.title(f'{torsion_type}: Multiplicity Distribution', fontsize=16)
    plt.grid(True, alpha=0.3, axis='y'); plt.tight_layout()
    plt.savefig(f'{output_dir}/({nmin},{nmax})_multiplicity_histogram.png', dpi=150); plt.show(); plt.close()
    
    # AP coefficients by rank
    print("Extracting ap coefficients...")
    rk_aps = [[], [], [], []]
    for iso in tqdm(seen_iso_classes):
        curve = curve_db[iso]
        rk = curve['rank']
        if rk < 4:
            rk_aps[rk].append(curve['ap_list'])
    
    rk_aps = [np.array(aps) if aps else None for aps in rk_aps]
    rk_avg = [np.mean(aps, axis=0) if aps is not None else None for aps in rk_aps]
    
    for i, aps in enumerate(rk_aps):
        if aps is not None:
            print(f"Rank {i}: {len(aps)} isogeny classes")
    
    # Save ap data
    averages_data = {f'rank_{i}_average': avg.tolist() if avg is not None else None for i, avg in enumerate(rk_avg)}
    averages_data.update({f'rank_{i}_count': len(aps) if aps is not None else 0 for i, aps in enumerate(rk_aps)})
    save(averages_data, f'{output_dir}/({nmin},{nmax})_average_aps.sobj')
    
    # AP plots
    N = len(rk_avg[0]) if rk_avg[0] is not None else (len(rk_avg[1]) if rk_avg[1] is not None else 0)
    MIN_SAMPLES = 100
    
    if N > 0:
        plt.figure(figsize=(14, 7))
        colors = ['C0', 'C1', 'C2', 'C3']
        for i, (aps, avg) in enumerate(zip(rk_aps, rk_avg)):
            if aps is not None and len(aps) >= MIN_SAMPLES:
                plt.scatter(range(N), avg, label=f"Rank {i} (n={len(aps)})", alpha=0.6, s=15, c=colors[i])
        plt.axhline(y=0, color='gray', linestyle='--', alpha=0.3)
        plt.legend(fontsize=12)
        plt.xlabel(r"Prime index $i$", fontsize=12); plt.ylabel(r"Average $a_{p_i}$", fontsize=12)
        plt.title(fr'{torsion_type}: Average $a_p$ by rank', fontsize=14)
        plt.grid(True, alpha=0.3); plt.tight_layout()
        plt.savefig(f'{output_dir}/({nmin},{nmax})_ap_averages.png', dpi=150); plt.show(); plt.close()

    # AP plot: Rank 0 vs Rank 1 only
    if N > 0 and rk_aps[0] is not None and rk_aps[1] is not None:
        if len(rk_aps[0]) >= MIN_SAMPLES and len(rk_aps[1]) >= MIN_SAMPLES:
            plt.figure(figsize=(14, 7))
            plt.scatter(range(N), rk_avg[0], label=f"Rank 0 (n={len(rk_aps[0])})", alpha=0.6, s=15, c='C0')
            plt.scatter(range(N), rk_avg[1], label=f"Rank 1 (n={len(rk_aps[1])})", alpha=0.6, s=15, c='C1')
            plt.axhline(y=0, color='gray', linestyle='--', alpha=0.3)
            plt.legend(fontsize=12)
            plt.xlabel(r"Prime index $i$", fontsize=12); plt.ylabel(r"Average $a_{p_i}$", fontsize=12)
            plt.title(fr'{torsion_type}: Average $a_p$ — Rank 0 vs Rank 1', fontsize=14)
            plt.grid(True, alpha=0.3); plt.tight_layout()
            plt.savefig(f'{output_dir}/({nmin},{nmax})_ap_averages_rank0_vs_rank1.png', dpi=150); plt.show(); plt.close()
    
    print(f"Done with {torsion_type}!")
    return scan_data

# Run for all torsion families
TORSION_FAMILIES = ['Z2', 'Z3', 'Z4', 'Z5', 'Z6', 'Z2+Z2', 'Z4+Z2']

for torsion in TORSION_FAMILIES:
    run_torsion_scan(torsion)

print("\nAll scans complete!")